In [ ]:
import mcfs

import numpy as np

import os
import json
from pathlib import Path
from fake_spectra.griddedspectra import GriddedSpectra

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = mcfs.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
# %matplotlib widget
%matplotlib inline

In [ ]:
# Point this to the manifest printed by your first notebook (info["manifest"])
base_load_path = "/home/STORAGE/SKEWERS"

simulation_name = "TNG50-2"
num             = 25
nspec           = 64
axis            = -1
nbins           = 2048
res_kms         = None

manifest_path = Path(os.path.join(base_load_path, simulation_name, f"snapdir_{num:03d}", f"grid_{simulation_name}_snap{num:03d}_nspec{nspec}_axis{axis}_nbins{nbins}.hdf5.json"))

with open(manifest_path, "r") as f:
    man = json.load(f)

hdf5_path = Path(man["hdf5_path"])
savedir   = str(hdf5_path.parent)
savefile  = hdf5_path.name

gs = GriddedSpectra(
    num=man["num"],
    base=man["base"],
    nspec=man["nspec"],
    axis=man["axis"],
    nbins=man["nbins"],
    res=man["res_kms"],
    savefile=savefile,
    savedir=savedir,
    reload_file=False,      # IMPORTANT: load existing file
    quiet=True,
)

print("Loaded:", hdf5_path)
print("z =", gs.red, "| NumLos =", gs.NumLos, "| nbins =", gs.nbins, "| dvbin =", gs.dvbin, "km/s")


In [ ]:
# --- choose lines (same as before) ---
lines = [
    ("H",  1, 1215),
    ("Si", 3, 1206),
    ("Si", 2, 1190),
    ("Si", 2, 1193),
]

# --- how many random skewers to plot ---
Nplot = nspec**2
rng = np.random.default_rng(137)
ii_list = rng.choice(gs.NumLos, size=min(Nplot, gs.NumLos), replace=False)

# --- x-axis in velocity units (km/s) ---
x = np.arange(gs.nbins) * gs.dvbin
xlabel = r"Distance along LOS [$\mathrm{km\,s^{-1}}$]"

# --- load tau for each line ---
tau = {}
keys = []
for elem, ion, lam in lines:
    key = f"{elem}{ion}_{lam}"
    tau[key] = gs.get_tau(elem, ion, lam)  # (NumLos, nbins)
    keys.append(key)

# --- print mean absorbed and mean flux across ALL skewers/pixels ---
print(f"Global means over all skewers and pixels (NumLos={gs.NumLos}, nbins={gs.nbins}):")
for k in keys:
    F = np.exp(-tau[k])
    mean_F = float(F.mean())
    print(f"  {k:10s}  <F> = {mean_F:.6f}")

# --- plotting ---
cmap = plt.get_cmap("tab10")
colors = {k: cmap(i % cmap.N) for i, k in enumerate(keys)}

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Panel 1: tau (plot multiple skewers with low alpha)
for k in keys:
    for ii in ii_list:
        axes[0].plot(x, tau[k][ii], color=colors[k], lw=1.0, alpha=0.7)
axes[0].set_ylabel(r"$\tau$")
axes[0].set_yscale("log")
axes[0].set_title(f"{len(ii_list)} random skewers | z={gs.red:.3f}")

# Panel 2: flux
for k in keys:
    for ii in ii_list:
        axes[1].plot(x, np.exp(-tau[k][ii]), color=colors[k], lw=1.0, alpha=0.7)
axes[1].set_ylabel(r"$F/F_0$")
axes[1].set_xlabel(xlabel)

# Legend: one entry per line (not per skewer)
handles = [Line2D([0], [0], color=colors[k], lw=3, label=k) for k in keys]
axes[0].legend(handles=handles, loc="upper right", frameon=True, fontsize=16, title="Line key")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations

from mcfs.p1d import p1d_auto_from_tau, p1d_cross_from_tau

# ---- define lines (same as before) ----
lines = [
    ("H",  1, 1215),
    ("Si", 3, 1206),
    ("Si", 2, 1190),
    ("Si", 2, 1193),
]

# ---- load tau arrays from gs ----
tau = {}
keys = []
for elem, ion, lam in lines:
    key = f"{elem}{ion}_{lam}"
    tau[key] = gs.get_tau(elem, ion, lam)   # (Nlos, nbins)
    keys.append(key)

vmax = gs.vmax

# ---- auto P1D ----
auto = {}
for k in keys:
    kk, pk = p1d_auto_from_tau(tau[k], vmax=vmax, mean_flux_desired=None)
    auto[k] = (kk, pk)

# ---- cross P1D (all unordered pairs) ----
cross = {}
for a, b in combinations(keys, 2):
    kk, pab = p1d_cross_from_tau(tau[a], tau[b], vmax=vmax)
    cross[(a, b)] = (kk, pab)

print("Computed auto P1D for:", keys)
print("Computed cross P1D for pairs:", list(cross.keys()))

In [ ]:
plt.figure(figsize=(8,5))
for k, (kk, pk) in auto.items():
    plt.loglog(kk[1:], pk[1:], label=k)  # skip k=0
plt.xlabel(r"$k$ [s/km]")
plt.ylabel(r"$P_{1D}(k)$")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
for (a, b), (kk, pab) in cross.items():
    plt.plot(kk[1:], pab[1:], label=f"{a} × {b}")
plt.xlabel(r"$k$ [s/km]")
plt.ylabel(r"$P^{\mathrm{cross}}_{1D}(k)$")
plt.axhline(0.0, lw=1)
plt.legend(fontsize=16, ncols=1)
plt.tight_layout()
plt.show()
